# 01 - The Self-Improving Copilot

The CityOps copilot you built in the prerequisite lab could *advise*. In this notebook it learns to *act* - and to get better at acting:

1. **Tool registry** - its capabilities become rows it retrieves by meaning
2. **Agent loop** - the model calls tools; every call is captured as a full trajectory
3. **Verified success** - an LLM judge audits each run against the database record; only verified runs count (no more `success = bool(final)`) 
4. **Workflow dedup** - recurring intents merge via an embedding gray band + LLM confirmation
5. **Skill harvest & lifecycle** - what recurs *and verifiably works* is distilled into a `SKILL.md`, born **provisional**, and earns **approved** (or **retired**) through linked outcomes

Each mechanism closes a gap documented in `memory_promotion_review.md`: promotion without curation, success without verification, growth without forgetting.

## 0 · Setup

Connect, verify notebook 00's artifacts, and stand up the in-database embedder.

In [ ]:
import array
import json
import uuid

from cityops_harness.checks import ok, check
from cityops_harness.config import load_settings
from cityops_harness.db import get_connection
from cityops_harness.llm import chat_model
from cityops_harness.state import require, verify
from cityops_harness.tracing import init_tracing
from cityops_harness import improve

settings = load_settings()
conn = get_connection(settings)
require(conn, "00_setup")
CHAT = chat_model(settings)
HANDLER = init_tracing(settings)
ok(f"connected - provider={settings.llm_provider}, langfuse={settings.langfuse_mode}")

In [ ]:
# The same IEmbedder bridge the CityOps lab uses: embeddings are computed
# inside the database via VECTOR_EMBEDDING() - no text egress, no API cost.
import asyncio
import numpy as np
from langchain_oracledb.embeddings.oracleai import OracleEmbeddings
from oracleagentmemory.apis.embedders.embedder import IEmbedder


class OracleONNXEmbedder(IEmbedder):
    def __init__(self, conn, model_name):
        self.provider = OracleEmbeddings(
            conn=conn, params={"provider": "database", "model": model_name}
        )

    def embed(self, texts, *, is_query=False):
        return np.asarray(self.provider.embed_documents(list(texts)), dtype=np.float32)

    async def embed_async(self, texts, *, is_query=False):
        return await asyncio.to_thread(self.embed, texts, is_query=is_query)


embedder = OracleONNXEmbedder(conn, settings.embed_model)
check(embedder.embed(["hello"]).shape == (1, 384), "in-database embedder returns 384-dim vectors")

## 1 · Domain bootstrap (pre-built)

The system of record from the CityOps lab, loaded idempotently: the asset registry and
~inspection findings with in-database embeddings. Re-running this section is safe.

In [ ]:
from pathlib import Path

_data_dir = Path("data") if Path("data").exists() else Path("../data")
with open(_data_dir / "maintenance_logs.json") as f:
    _logs = json.load(f)
with open(_data_dir / "inspection_reports.json") as f:
    _reports = json.load(f)
_asset_names = sorted({x["asset_name"] for x in _logs} | {x["asset_name"] for x in _reports})


def classify_asset(name: str) -> str:
    n = name.lower()
    for key, cls in (("bridge", "bridge"), ("tower", "tower"), ("pump", "pump_station"),
                     ("tunnel", "tunnel"), ("plant", "plant"), ("station", "station")):
        if key in n:
            return cls
    return "structure"


def _table_exists(name: str) -> bool:
    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM user_tables WHERE table_name = :t", t=name)
        return cur.fetchone()[0] > 0


if not _table_exists("CITY_ASSET"):
    with conn.cursor() as cur:
        cur.execute("""CREATE TABLE CITY_ASSET (
            asset_id     VARCHAR2(128) PRIMARY KEY,
            asset_class  VARCHAR2(32) NOT NULL,
            created_at   TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""")
        cur.executemany("INSERT INTO CITY_ASSET (asset_id, asset_class) VALUES (:1, :2)",
                        [(a, classify_asset(a)) for a in _asset_names])
    conn.commit()


def get_asset(asset_id: str):
    with conn.cursor() as cur:
        cur.execute("SELECT asset_id, asset_class FROM CITY_ASSET WHERE asset_id = :id", id=asset_id)
        row = cur.fetchone()
    return {"asset_id": row[0], "asset_class": row[1]} if row else None

ok(f"{len(_asset_names)} assets registered")

In [ ]:
import os

DEMO_FINDING_COUNT = int(os.getenv("DEMO_FINDING_COUNT", "60"))

if not _table_exists("CITY_INSPECTION_FINDING"):
    with conn.cursor() as cur:
        cur.execute("""CREATE TABLE CITY_INSPECTION_FINDING (
            finding_id      VARCHAR2(64) PRIMARY KEY,
            asset_id        VARCHAR2(128) NOT NULL,
            inspector       VARCHAR2(128),
            overall_grade   VARCHAR2(2),
            category        VARCHAR2(32),
            severity        VARCHAR2(16),
            description     CLOB NOT NULL,
            recommendation  CLOB,
            days_ago        NUMBER,
            embedding       VECTOR(384) NOT NULL,
            created_at      TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            CONSTRAINT fk_finding_asset FOREIGN KEY (asset_id) REFERENCES CITY_ASSET(asset_id))""")
        try:
            cur.execute("""CREATE VECTOR INDEX city_finding_embedding_idx
                ON CITY_INSPECTION_FINDING (embedding)
                ORGANIZATION INMEMORY NEIGHBOR GRAPH DISTANCE COSINE
                WITH TARGET ACCURACY 95 PARAMETERS (TYPE HNSW, M 16, EFCONSTRUCTION 200)""")
        except Exception as e:
            print(f"  (skipped HNSW index: {e})")
    conn.commit()

with conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM CITY_INSPECTION_FINDING")
    _existing = cur.fetchone()[0]

if _existing == 0:
    rows = []
    for report in _reports:
        if len(rows) >= DEMO_FINDING_COUNT:
            break
        for finding in report["findings"]:
            if len(rows) >= DEMO_FINDING_COUNT:
                break
            vec = array.array('f', embedder.embed([finding["description"]])[0].tolist())
            rows.append({
                "finding_id": str(uuid.uuid4())[:12], "asset_id": report["asset_name"],
                "inspector": report["inspector"], "overall_grade": report["overall_grade"],
                "category": finding["category"], "severity": finding["severity"],
                "description": finding["description"], "recommendation": finding["recommendation"],
                "days_ago": report["days_ago"], "embedding": vec,
            })
    with conn.cursor() as cur:
        cur.executemany("""INSERT INTO CITY_INSPECTION_FINDING
            (finding_id, asset_id, inspector, overall_grade, category, severity,
             description, recommendation, days_ago, embedding)
            VALUES (:finding_id, :asset_id, :inspector, :overall_grade, :category, :severity,
                    :description, :recommendation, :days_ago, :embedding)""", rows)
    conn.commit()
    _existing = len(rows)
ok(f"{_existing} findings in the system of record")

In [ ]:
def log_finding(asset_id, inspector, category, severity, description,
                recommendation="", overall_grade=None, days_ago=0):
    finding_id = str(uuid.uuid4())[:12]
    vec = array.array('f', embedder.embed([description])[0].tolist())
    with conn.cursor() as cur:
        cur.execute("""INSERT INTO CITY_INSPECTION_FINDING
             (finding_id, asset_id, inspector, overall_grade, category, severity,
              description, recommendation, days_ago, embedding)
           VALUES (:finding_id, :asset_id, :inspector, :overall_grade, :category, :severity,
                   :description, :recommendation, :days_ago, :embedding)""",
            finding_id=finding_id, asset_id=asset_id, inspector=inspector,
            overall_grade=overall_grade, category=category, severity=severity,
            description=description, recommendation=recommendation,
            days_ago=days_ago, embedding=vec)
    conn.commit()
    return finding_id


def find_similar_findings(description, asset_id=None, category=None, k=3):
    query_vec = array.array('f', embedder.embed([description])[0].tolist())
    sql = """SELECT finding_id, asset_id, inspector, overall_grade, category, severity,
                    description, recommendation, days_ago,
                    VECTOR_DISTANCE(embedding, :q, COSINE) AS score
               FROM CITY_INSPECTION_FINDING
              WHERE (:asset_id IS NULL OR asset_id = :asset_id)
                AND (:category IS NULL OR category = :category)
              ORDER BY score FETCH FIRST :k ROWS ONLY"""
    with conn.cursor() as cur:
        cur.execute(sql, q=query_vec, asset_id=asset_id, category=category, k=k)
        cols = [d[0].lower() for d in cur.description]
        out = []
        for r in cur.fetchall():
            row = dict(zip(cols, r))
            for key in ("description", "recommendation"):
                v = row.get(key)
                if v is not None and hasattr(v, "read"):
                    row[key] = v.read()
            out.append(row)
    return out

ASSET_A = "Harbor Bridge" if "Harbor Bridge" in _asset_names else _asset_names[0]
# Pick a second asset of the SAME class as ASSET_A (not just alphabetically-next) so
# the demo's structural-corrosion task actually makes sense against it - the naive
# 'first other name' pick can land on an unrelated asset (e.g. a sensor), and the
# model then declines to log a nonsensical finding or invents an asset_id, which
# violates the CITY_INSPECTION_FINDING -> CITY_ASSET foreign key.
_a_class = classify_asset(ASSET_A)
ASSET_B = (
    next((a for a in _asset_names if a != ASSET_A and classify_asset(a) == _a_class
          and "Sensor" not in a), None)
    or next((a for a in _asset_names if a != ASSET_A and classify_asset(a) == _a_class), None)
    or next(a for a in _asset_names if a != ASSET_A)
)
ok(f"domain tools ready - demo assets: {ASSET_A!r}, {ASSET_B!r}")

## 2 · The harness registries

Three plain tables the harness *writes to as it learns*:

- `HARNESS_TOOLS` - capabilities, retrieved by meaning
- `HARNESS_WORKFLOW` - captured trajectories with **verified** outcome counts
- `HARNESS_SKILLS` - distilled `SKILL.md` documents with a **lifecycle** (provisional → approved | retired), linked back to outcomes via `skill_id`

In [ ]:
for ddl in [
    """CREATE TABLE HARNESS_TOOLS (
        name           VARCHAR2(64) PRIMARY KEY,
        description    VARCHAR2(1000) NOT NULL,
        params_schema  CLOB,
        embedding      VECTOR(384) NOT NULL,
        created_at     TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""",
    """CREATE TABLE HARNESS_WORKFLOW (
        workflow_id        VARCHAR2(64) PRIMARY KEY,
        intent             VARCHAR2(2000) NOT NULL,
        intent_embedding   VECTOR(384) NOT NULL,
        steps              CLOB NOT NULL,
        tools_used         VARCHAR2(1000),
        occurrences        NUMBER DEFAULT 1,
        verified_successes NUMBER DEFAULT 0,
        failures           NUMBER DEFAULT 0,
        promoted           VARCHAR2(1) DEFAULT 'N',
        last_seen          TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        created_at         TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""",
    """CREATE TABLE HARNESS_SKILLS (
        skill_id           VARCHAR2(64) PRIMARY KEY,
        name               VARCHAR2(128) NOT NULL,
        description        VARCHAR2(1000) NOT NULL,
        content            CLOB NOT NULL,
        sha                VARCHAR2(64) NOT NULL,
        status             VARCHAR2(16) DEFAULT 'provisional'
                           CHECK (status IN ('provisional','approved','retired')),
        schema_sha         VARCHAR2(64),
        source_workflow_id VARCHAR2(64),
        embedding          VECTOR(384) NOT NULL,
        uses               NUMBER DEFAULT 0,
        verified_successes NUMBER DEFAULT 0,
        failures           NUMBER DEFAULT 0,
        created_at         TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        updated_at         TIMESTAMP DEFAULT CURRENT_TIMESTAMP)""",
]:
    table = ddl.split("CREATE TABLE ")[1].split(" ")[0].strip()
    if not _table_exists(table):
        with conn.cursor() as cur:
            cur.execute(ddl)
        conn.commit()
ok("HARNESS_TOOLS / HARNESS_WORKFLOW / HARNESS_SKILLS ready")

In [ ]:
def current_schema_sha():
    """Fingerprint of the domain schema a skill was distilled against.
    Skills whose stored schema_sha no longer matches are flagged stale at retrieval."""
    with conn.cursor() as cur:
        cur.execute("""SELECT table_name, column_name, data_type
                         FROM user_tab_columns
                        WHERE table_name LIKE 'CITY!_%' ESCAPE '!'""")
        cols = cur.fetchall()
    return improve.compute_schema_sha(cols)

SCHEMA_SHA = current_schema_sha()
ok(f"schema fingerprint {SCHEMA_SHA[:12]}...")

## 3 · The toolbox: capabilities retrieved by meaning

Each LangChain tool is registered with an embedded description. At run time the
agent loop retrieves only the tools relevant to the task - **with a distance
threshold**, so an off-topic task injects nothing (one of the review's fixes).

In [ ]:
from langchain_core.tools import tool

CURRENT_INSPECTOR = "R_Mercer"


@tool
def tool_find_similar_findings(description: str, asset_id: str = None, k: int = 3) -> str:
    """Search past inspection findings semantically similar to a description.
    Returns finding_id, inspector, severity, recommendation and distance per hit."""
    hits = find_similar_findings(description, asset_id=asset_id, k=k)
    return json.dumps([
        {k2: v for k2, v in h.items() if k2 != 'embedding'} for h in hits
    ], default=str)


@tool
def tool_log_finding(asset_id: str, category: str, severity: str,
                     description: str, recommendation: str = "") -> str:
    """Record a new inspection finding in the system of record.
    Returns the new finding_id. Use only after checking similar past findings."""
    return log_finding(asset_id=asset_id, inspector=CURRENT_INSPECTOR, category=category,
                       severity=severity, description=description, recommendation=recommendation)


@tool
def tool_get_asset(asset_id: str) -> str:
    """Look up an asset's registry record (asset_id, asset_class)."""
    return json.dumps(get_asset(asset_id))


TOOLBOX = {t.name: t for t in [tool_find_similar_findings, tool_log_finding, tool_get_asset]}
ok(f"{len(TOOLBOX)} tools defined")

In [ ]:
def register_tool(t):
    text = f"{t.name}: {t.description}"
    vec = array.array('f', embedder.embed([text])[0].tolist())
    with conn.cursor() as cur:
        cur.execute("DELETE FROM HARNESS_TOOLS WHERE name = :n", n=t.name)
        cur.execute("""INSERT INTO HARNESS_TOOLS (name, description, params_schema, embedding)
                       VALUES (:n, :d, :p, :e)""",
                    n=t.name, d=t.description, p=json.dumps(t.args), e=vec)
    conn.commit()


for _t in TOOLBOX.values():
    register_tool(_t)

# Calibrated against the live ALL_MINILM_L12_V2 in-database embedder: cosine
# distances between short generic tool descriptions and a task sentence run
# noticeably higher than one might expect (~0.75-0.98 for genuinely relevant
# tools here - even higher once a skill-using task's wording drifts from the
# tools' own descriptions), so too low a cutoff starves the agent of its own
# tools. 0.92 still keeps a clear margin below off-topic distances (>=0.939).
TOOL_DISTANCE_MAX = 0.92


def retrieve_tools(query: str, k: int = 4):
    qv = array.array('f', embedder.embed([query])[0].tolist())
    with conn.cursor() as cur:
        cur.execute("""SELECT name, description, VECTOR_DISTANCE(embedding, :q, COSINE) AS distance
                         FROM HARNESS_TOOLS ORDER BY distance FETCH FIRST :k ROWS ONLY""",
                    q=qv, k=k)
        rows = [dict(zip([d[0].lower() for d in cur.description], r)) for r in cur.fetchall()]
    # ✏️ TODO(1): apply the distance threshold so irrelevant tools are never injected.
    #             Use improve.filter_by_distance(rows, TOOL_DISTANCE_MAX).
    # TODO-SOLUTION-START
    rows = improve.filter_by_distance(rows, TOOL_DISTANCE_MAX)
    # TODO-SOLUTION-END
    return rows


_hits = retrieve_tools("corrosion on bridge bearings - check history and record it")
check(len(_hits) >= 2, f"task-relevant retrieval: {[h['name'] for h in _hits]}")
check(retrieve_tools("what is the weather tomorrow in Paris?") == [],
      "off-topic query injects no tools")

## 4 · The agent loop: act, and remember *exactly* what you did

The review's sharpest finding: the original harness captured a **sorted set of tool
names** - nothing to distill from. Here every step records the tool, its **arguments**,
and a **truncated result**, in order. That trajectory is what the judge audits and
what skills are distilled from.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

AGENT_SYSTEM_PROMPT = (
    "You are the CityOps inspection copilot. You act through tools.\n"
    "Rules: check similar past findings BEFORE recording a new one; base severity on\n"
    "evidence; when you record a finding, cite its finding_id in your answer.\n"
    "Your final summary will be audited against the tool call results above - state\n"
    "only facts, figures, and comparisons that actually appear in those results; do not\n"
    "add estimates, trends, or specifics you did not retrieve.\n"
    "Finish with a concise summary (<= 6 sentences)."
)


def run_copilot_task(task: str, inspector: str = "R_Mercer", use_skills: bool = True,
                     max_iters: int = 6) -> dict:
    global CURRENT_INSPECTOR
    CURRENT_INSPECTOR = inspector
    manifest, skill_ids = (build_skill_manifest(task) if use_skills else ("", []))
    tools_meta = retrieve_tools(task)
    lc_tools = [TOOLBOX[t["name"]] for t in tools_meta if t["name"] in TOOLBOX]
    model = CHAT.bind_tools(lc_tools) if lc_tools else CHAT
    cfg = {"callbacks": [HANDLER]} if HANDLER else {}
    msgs = [SystemMessage(content=AGENT_SYSTEM_PROMPT + manifest), HumanMessage(content=task)]
    trajectory = []
    resp = None
    for _ in range(max_iters):
        resp = model.invoke(msgs, config=cfg)
        msgs.append(resp)
        if not getattr(resp, "tool_calls", None):
            break
        for tc in resp.tool_calls:
            try:
                result = str(TOOLBOX[tc["name"]].invoke(tc["args"]))
            except Exception as e:
                # A tool error becomes ToolMessage content, not a crashed run: the
                # model sees it and can recover, and the judge sees it in the trajectory.
                result = f"ERROR: {type(e).__name__}: {e}"
            # ✏️ TODO(2): capture this step for the trajectory - append a dict with
            #             the tool name, its args, and improve.truncate_result(result).
            # TODO-SOLUTION-START
            trajectory.append({"tool": tc["name"], "args": tc["args"],
                               "result": improve.truncate_result(result)})
            # TODO-SOLUTION-END
            msgs.append(ToolMessage(content=result, tool_call_id=tc["id"]))
    answer = resp.content if resp is not None else ""
    if isinstance(answer, list):
        # Some providers return content as a list of content-block dicts rather than
        # a plain string; flatten to text so downstream string ops don't break.
        answer = "".join(
            p.get("text", "") if isinstance(p, dict) else str(p) for p in answer
        )
    return {"answer": answer, "trajectory": trajectory, "skill_ids": skill_ids}


def build_skill_manifest(query):  # placeholder until Section 6 redefines it
    return "", []

ok("agent loop defined")

## 5 · Verified success: the judge audits the database, not the prose

`success = bool(final)` let confident nonsense count as success. Here an LLM judge
sees the task, the full trajectory, the final answer, **and the database evidence**
(claimed writes are re-queried). Only a verified run feeds the workflow's success count -
and every verdict is logged to Langfuse as a score when tracing is on.

In [ ]:
from pydantic import BaseModel


class Verdict(BaseModel):
    verified: bool
    reason: str


JUDGE_PROMPT = (
    "You are auditing one run of an inspection copilot against the database record.\n"
    "Mark verified=true if ALL of these hold:\n"
    "  (a) the trajectory shows the copilot did what the task required (searched history\n"
    "      before recording; recorded a finding when the task asked for one);\n"
    "  (b) every database write it claims is confirmed by the evidence below;\n"
    "  (c) every finding_id cited in the final answer appears in the evidence as EXISTS;\n"
    "  (d) the final answer does not contradict the trajectory or invent database records.\n"
    "Interpretive judgement (severity reasoning, recommendations, comparisons phrased as\n"
    "the copilot's own assessment) is allowed and does NOT fail the audit. Fabricated\n"
    "records, fabricated finding_ids, or claims of actions absent from the trajectory DO\n"
    "fail it.\n\n"
    "# Task\n{task}\n\n# Trajectory\n{trajectory}\n\n# Final answer\n{answer}\n\n"
    "# Database evidence\n{evidence}"
)


def build_evidence(trajectory, answer=""):
    lines = []
    for step in trajectory:
        if step["tool"] == "tool_log_finding":
            fid = step["result"].strip()
            with conn.cursor() as cur:
                cur.execute("""SELECT asset_id, category, severity FROM CITY_INSPECTION_FINDING
                               WHERE finding_id = :f""", f=fid)
                row = cur.fetchone()
            if row:
                lines.append(f"finding {fid} EXISTS: asset={row[0]} category={row[1]} severity={row[2]}")
            else:
                lines.append(f"finding {fid} NOT FOUND in CITY_INSPECTION_FINDING")
    import re
    for fid in sorted(set(re.findall(r"\b[0-9a-f]{8}-[0-9a-f]{3}\b", answer))):
        with conn.cursor() as cur:
            cur.execute("SELECT COUNT(*) FROM CITY_INSPECTION_FINDING WHERE finding_id = :f", f=fid)
            n = cur.fetchone()[0]
        lines.append(f"cited finding {fid} {'EXISTS' if n else 'NOT FOUND'}")
    return "\n".join(lines) or "(no database writes claimed)"


_judge_model = CHAT.with_structured_output(Verdict)


def judge_workflow(task: str, run: dict) -> Verdict:
    prompt = JUDGE_PROMPT.format(
        task=task,
        trajectory=improve.trajectory_to_text(run["trajectory"]),
        answer=run["answer"],
        evidence=build_evidence(run["trajectory"], run["answer"]),
    )
    cfg = {"callbacks": [HANDLER]} if HANDLER else {}
    return _judge_model.invoke(prompt, config=cfg)


def record_score(name: str, value: float, comment: str = "", trace_id: str | None = None):
    """Log a verdict as a Langfuse score, attached to the run's own trace so it is
    not orphaned. When `trace_id` is given (the id of the trace opened by
    run_and_learn's span, see Section 6) it is passed straight to create_score;
    otherwise falls back to scoring whatever trace is currently active. Silently
    no-ops when tracing is off."""
    if HANDLER is None:
        return
    try:
        from langfuse import get_client
        client = get_client()
        if trace_id is not None:
            client.create_score(trace_id=trace_id, name=name, value=value, comment=comment)
        else:
            client.score_current_trace(name=name, value=value, comment=comment)
        client.flush()
    except Exception as e:
        print(f"(langfuse score skipped: {e})")

ok("judge ready")

## 6 · Workflow capture: dedup by meaning, with an honest gray band

A fixed 0.15 cosine cutoff under-merges paraphrases and over-merges neighbours.
Here distances split three ways: certain merge, certain new, and a **gray band**
where a cheap LLM call answers 'same task?' before counts are polluted.

One residual limitation stays open, on purpose: a phrasing beyond the `new` boundary
still splits silently - the gray band narrows the failure mode, it does not eliminate
it. Notebook 04's evals measure how often that actually happens.

In [ ]:
class SameTask(BaseModel):
    same: bool


_same_task_model = CHAT.with_structured_output(SameTask)


def _nearest_workflow(intent_vec):
    with conn.cursor() as cur:
        cur.execute("""SELECT workflow_id, intent,
                              VECTOR_DISTANCE(intent_embedding, :q, COSINE) AS distance
                         FROM HARNESS_WORKFLOW ORDER BY distance FETCH FIRST 1 ROWS ONLY""",
                    q=intent_vec)
        row = cur.fetchone()
    if row is None:
        return None
    return {"workflow_id": row[0],
            "intent": row[1].read() if hasattr(row[1], "read") else row[1],
            "distance": row[2]}


def capture_workflow(intent: str, run: dict, verified: bool) -> str:
    intent_vec = array.array('f', embedder.embed([intent])[0].tolist())
    nearest = _nearest_workflow(intent_vec)
    decision = "new"
    if nearest is not None:
        # ✏️ TODO(3): decide merge/new. Use improve.merge_decision(nearest['distance']);
        #             on 'ask', confirm with the LLM: are `intent` and `nearest['intent']`
        #             the same recurring task? (_same_task_model.invoke(...).same)
        # TODO-SOLUTION-START
        decision = improve.merge_decision(nearest["distance"])
        if decision == "ask":
            q = (f"Task A: {intent}\nTask B: {nearest['intent']}\n"
                 "Are these the same recurring task type (same procedure, possibly "
                 "different asset)? Answer same=true/false.")
            decision = "merge" if _same_task_model.invoke(q).same else "new"
        # TODO-SOLUTION-END
    if decision == "merge":
        wid = nearest["workflow_id"]
        with conn.cursor() as cur:
            # ✏️ TODO(4): only a VERIFIED run may increment verified_successes; an
            #             unverified run increments failures. Update occurrences,
            #             the right counter, steps (latest verified trajectory), last_seen.
            # TODO-SOLUTION-START
            if verified:
                cur.execute("""UPDATE HARNESS_WORKFLOW
                               SET occurrences = occurrences + 1,
                                   verified_successes = verified_successes + 1,
                                   steps = :s, intent_embedding = :e,
                                   last_seen = CURRENT_TIMESTAMP
                             WHERE workflow_id = :w""",
                            s=json.dumps(run["trajectory"], default=str), e=intent_vec, w=wid)
            else:
                cur.execute("""UPDATE HARNESS_WORKFLOW
                               SET occurrences = occurrences + 1,
                                   failures = failures + 1,
                                   intent_embedding = :e,
                                   last_seen = CURRENT_TIMESTAMP
                             WHERE workflow_id = :w""", e=intent_vec, w=wid)
            # TODO-SOLUTION-END
        conn.commit()
        return wid
    wid = str(uuid.uuid4())[:12]
    with conn.cursor() as cur:
        cur.execute("""INSERT INTO HARNESS_WORKFLOW
                       (workflow_id, intent, intent_embedding, steps, tools_used,
                        verified_successes, failures)
                       VALUES (:w, :i, :e, :s, :t, :vs, :f)""",
                    w=wid, i=intent, e=intent_vec,
                    s=json.dumps(run["trajectory"], default=str),
                    t=",".join(sorted({s["tool"] for s in run["trajectory"]})),
                    vs=1 if verified else 0, f=0 if verified else 1)
    conn.commit()
    return wid


def run_and_learn(task: str, **kwargs) -> dict:
    """One full cycle: act -> judge -> record verdict -> capture -> update skill outcomes.

    When tracing is on, the whole cycle runs inside one Langfuse span so every model
    call made by run_copilot_task and judge_workflow - via HANDLER, an ambient OTel
    callback - lands as a child observation of that span, and the verdict score is
    attached to that SAME trace (langfuse==4.14.1's LangchainCallbackHandler has no
    metadata['langfuse_trace_id'] hook; start_as_current_observation()/
    score_current_trace() is the mechanism the installed SDK actually exposes)."""
    if HANDLER is not None:
        from langfuse import get_client
        with get_client().start_as_current_observation(
            name="run_and_learn", as_type="span"
        ) as _span:
            tid = _span.trace_id
            run = run_copilot_task(task, **kwargs)
            verdict = judge_workflow(task, run)
            record_score("workflow_verified", 1.0 if verdict.verified else 0.0,
                         verdict.reason, trace_id=tid)
    else:
        run = run_copilot_task(task, **kwargs)
        verdict = judge_workflow(task, run)
        record_score("workflow_verified", 1.0 if verdict.verified else 0.0, verdict.reason)
    wid = capture_workflow(task, run, verdict.verified)
    update_skill_outcomes(run["skill_ids"], verdict.verified)
    print(f"verified={verdict.verified} workflow={wid}\n  judge: {verdict.reason[:140]}")
    return {"run": run, "verdict": verdict, "workflow_id": wid}


def update_skill_outcomes(skill_ids, verified):  # redefined in Section 7
    pass

ok("capture pipeline ready")

## 7 · Harvesting skills - and making them earn their authority

The harvester promotes workflows that **recur and verifiably work** into `SKILL.md`
documents. Three review fixes land here:

- distillation reads the **full trajectory** (args + results), then a **faithfulness
  check** compares the distilled steps back against it - unfaithful skills are rejected
- a new skill is born **provisional** (low prompt authority) and records the
  **schema fingerprint** it was distilled against
- retrieval applies a **distance threshold**, injects **approved before provisional**,
  and flags **stale** skills whose schema no longer matches

In [ ]:
class Distilled(BaseModel):
    name: str
    description: str
    when_to_use: str
    steps_body: str


class Faithful(BaseModel):
    faithful: bool
    reason: str


_distill_model = CHAT.with_structured_output(Distilled)
_faithful_model = CHAT.with_structured_output(Faithful)


def promote_workflow_to_skill(row) -> str | None:
    steps_text = improve.trajectory_to_text(json.loads(row["steps"]))
    d = _distill_model.invoke(
        "Distill this verified, recurring copilot trajectory into a reusable procedure.\n"
        "Ground every step in what ACTUALLY happened - name the tools used, in order,\n"
        "with parameterised arguments. Do not invent steps.\n\n"
        f"Intent: {row['intent']}\n\nTrajectory:\n{steps_text}"
    )
    f = _faithful_model.invoke(
        "Do these distilled steps faithfully describe this trajectory - same tools, same\n"
        "order, no invented actions? faithful=true/false.\n\n"
        f"Distilled steps:\n{d.steps_body}\n\nTrajectory:\n{steps_text}"
    )
    if not f.faithful:
        print(f"  REJECTED (unfaithful distillation): {f.reason[:120]}")
        return None
    tools_used = sorted({s['tool'] for s in json.loads(row['steps'])})
    content = improve.render_skill_md(
        name=d.name, description=d.description, tools=tools_used,
        when_to_use=d.when_to_use, steps_body=d.steps_body,
        source_workflow_id=row["workflow_id"], schema_sha=SCHEMA_SHA)
    import hashlib
    sha = hashlib.sha256(content.encode()).hexdigest()
    sid = str(uuid.uuid4())[:12]
    vec = array.array('f', embedder.embed([f"{d.name}: {d.description}"])[0].tolist())
    with conn.cursor() as cur:
        cur.execute("""INSERT INTO HARNESS_SKILLS
                       (skill_id, name, description, content, sha, status,
                        schema_sha, source_workflow_id, embedding)
                       VALUES (:sid, :n, :d, :c, :sha, 'provisional', :ss, :src, :e)""",
                    sid=sid, n=d.name, d=d.description, c=content, sha=sha,
                    ss=SCHEMA_SHA, src=row["workflow_id"], e=vec)
        cur.execute("UPDATE HARNESS_WORKFLOW SET promoted = 'Y' WHERE workflow_id = :w",
                    w=row["workflow_id"])
    conn.commit()
    print(f"  harvested PROVISIONAL skill {d.name!r} ({sid}) from workflow {row['workflow_id']}")
    return sid


def harvest() -> list:
    with conn.cursor() as cur:
        cur.execute("""SELECT workflow_id, intent, steps, occurrences,
                              verified_successes, failures
                         FROM HARNESS_WORKFLOW WHERE promoted = 'N'""")
        cols = [d[0].lower() for d in cur.description]
        rows = []
        for r in cur.fetchall():
            row = dict(zip(cols, r))
            if hasattr(row["steps"], "read"):
                row["steps"] = row["steps"].read()
            if hasattr(row["intent"], "read"):
                row["intent"] = row["intent"].read()
            rows.append(row)
    harvested = []
    for row in rows:
        # ✏️ TODO(5): the harvest gate - promote only what recurs AND verifiably works.
        #             Use improve.harvest_ready(occurrences, verified_successes, failures).
        # TODO-SOLUTION-START
        if not improve.harvest_ready(row["occurrences"], row["verified_successes"],
                                     row["failures"]):
            continue
        # TODO-SOLUTION-END
        sid = promote_workflow_to_skill(row)
        if sid:
            harvested.append(sid)
    return harvested

ok("harvester ready")

In [ ]:
# Calibrated the same way as TOOL_DISTANCE_MAX: measured live, a task phrased
# differently from the skill's distilled name/description can sit just past a
# tight cutoff (observed 0.559 for a genuinely matching task at 0.55), while an
# off-topic query sits far away (observed >=1.0). 0.65 keeps a wide margin both
# ways.
SKILL_DISTANCE_MAX = 0.65


def retrieve_skills(query: str, k: int = 3):
    qv = array.array('f', embedder.embed([query])[0].tolist())
    with conn.cursor() as cur:
        cur.execute("""SELECT skill_id, name, status, content, schema_sha,
                              VECTOR_DISTANCE(embedding, :q, COSINE) AS distance
                         FROM HARNESS_SKILLS WHERE status != 'retired'
                        ORDER BY distance FETCH FIRST :k ROWS ONLY""", q=qv, k=k)
        cols = [d[0].lower() for d in cur.description]
        rows = []
        for r in cur.fetchall():
            row = dict(zip(cols, r))
            if hasattr(row["content"], "read"):
                row["content"] = row["content"].read()
            rows.append(row)
    return improve.filter_by_distance(rows, SKILL_DISTANCE_MAX)


def build_skill_manifest(query):
    """Two-level retrieval: approved skills carry authority; provisional ones are
    labelled as unproven; stale schema fingerprints are flagged."""
    hits = retrieve_skills(query)
    if not hits:
        return "", []
    cur_sha = current_schema_sha()
    parts, ids = [], []
    for h in sorted(hits, key=lambda r: 0 if r["status"] == "approved" else 1):
        ids.append(h["skill_id"])
        stale = " [STALE: schema changed since distillation - re-verify]" \
            if h["schema_sha"] != cur_sha else ""
        if h["status"] == "approved":
            label = f"APPROVED procedure{stale} - prefer this approach:"
        else:
            label = (f"PROVISIONAL procedure{stale} - an unproven candidate; treat as a"
                     " suggestion and verify against fresh data:")
        parts.append(f"\n\n--- {label} ---\n{h['content']}")
    return "".join(parts), ids


def update_skill_outcomes(skill_ids, verified):
    """Close the loop the review said never closes: runs that used a skill feed the
    skill's own record, and the lifecycle transition runs on the new counts."""
    for sid in skill_ids:
        with conn.cursor() as cur:
            cur.execute("""UPDATE HARNESS_SKILLS SET uses = uses + 1,
                           verified_successes = verified_successes + :vs,
                           failures = failures + :f, updated_at = CURRENT_TIMESTAMP
                         WHERE skill_id = :sid""",
                        vs=1 if verified else 0, f=0 if verified else 1, sid=sid)
            cur.execute("""SELECT status, verified_successes, failures
                             FROM HARNESS_SKILLS WHERE skill_id = :sid""", sid=sid)
            status, vs, fl = cur.fetchone()
            new_status = improve.lifecycle_transition(status, vs, fl)
            if new_status != status:
                cur.execute("UPDATE HARNESS_SKILLS SET status = :s WHERE skill_id = :sid",
                            s=new_status, sid=sid)
                print(f"  skill {sid}: {status} -> {new_status}")
        conn.commit()

ok("skill retrieval + lifecycle ready")

## 8 · The arc: recur → verify → harvest → earn approval

Three inspectors phrase the same corrosion-triage task differently. Watch the gray
band merge them, the judge verify them, and the harvester fire at three verified
occurrences. Then a fourth task *uses* the provisional skill - and its verified
outcomes promote it to approved.

In [ ]:
print(f"=== Occurrence 1 ({ASSET_A}) ===")
r1 = run_and_learn(
    f"Inspect {ASSET_A}: heavy corrosion on the bearing plates at pier 2. Assess severity"
    " against past findings and record a finding with your recommendation.",
    use_skills=False)

In [ ]:
print(f"=== Occurrence 2 ({ASSET_A}, new phrasing) ===")
r2 = run_and_learn(
    f"Corrosion check on {ASSET_A} bearings - compare with prior reports and log what"
    " you find with a recommendation.",
    inspector="T_Vance", use_skills=False)

In [ ]:
print(f"=== Occurrence 3 ({ASSET_B}, same task type, different asset) ===")
r3 = run_and_learn(
    f"Inspect {ASSET_B}: surface rust and section loss on the bearing members. Assess"
    " severity against past findings and record a finding with your recommendation.",
    inspector="K_Osei", use_skills=False)

In [ ]:
# The registry after three occurrences: one workflow row, merged by meaning.
with conn.cursor() as cur:
    cur.execute("""SELECT workflow_id, occurrences, verified_successes, failures, promoted
                     FROM HARNESS_WORKFLOW""")
    for row in cur.fetchall():
        print(row)

In [ ]:
print("=== Harvest ===")
new_skills = harvest()
check(len(new_skills) >= 1, f"harvested {len(new_skills)} provisional skill(s)")
with conn.cursor() as cur:
    cur.execute("SELECT skill_id, name, status FROM HARNESS_SKILLS")
    for row in cur.fetchall():
        print(row)

In [ ]:
# Show the distilled SKILL.md - grounded in the real trajectory, not confabulated.
with conn.cursor() as cur:
    cur.execute("SELECT content FROM HARNESS_SKILLS FETCH FIRST 1 ROWS ONLY")
    _content = cur.fetchone()[0]
print(_content.read() if hasattr(_content, 'read') else _content)

In [ ]:
print(f"=== Using the provisional skill (run 1) ===")
r4 = run_and_learn(
    f"New corrosion report on {ASSET_A}: pitting on expansion-joint anchor bolts."
    " Triage against history and record a finding.",
    inspector="J_Ibarra")
check(len(r4["run"]["skill_ids"]) >= 1, "provisional skill was retrieved and used")

In [ ]:
print(f"=== Using the skill again (run 2) - promotion happens here if both verified ===")
r5 = run_and_learn(
    f"{ASSET_B}: corrosion staining under deck drainage outlets. Check history, then"
    " record a finding with recommendation.",
    inspector="R_Mercer")
with conn.cursor() as cur:
    cur.execute("SELECT skill_id, name, status, uses, verified_successes FROM HARNESS_SKILLS")
    for row in cur.fetchall():
        print(row)

In [ ]:
# Retrieval thresholds in action: an off-topic task injects NO skills, no tools.
manifest, ids = build_skill_manifest("plan the office holiday party")
check(manifest == "" and ids == [], "off-topic query injects no skills")

## 9 · What you built (vs. what the review criticised)

| Review gap | This notebook |
|---|---|
| Trajectory = sorted set of tool names | Ordered steps with args + truncated results |
| `success = bool(final)` | LLM judge audits trajectory **and database evidence** |
| Brittle 0.15 dedup cutoff | Gray band + LLM same-task confirmation |
| One-way promotion, max authority | Provisional → approved/retired via linked outcomes |
| No staleness handling | Schema fingerprint checked at retrieval |
| Unconditional top-k injection | Distance thresholds on tools *and* skills |

Next: **02 - Scheduled briefings**, where the *other* promotion path (scratch →
long-term memory) gets the same treatment: opt-in curation, provenance, supersession,
and a consumer that is actually scheduled.

In [ ]:
for desc, passed in verify(conn, "01_self_improving_copilot"):
    check(passed, desc)
ok("notebook 01 artifacts in place - continue to 02_scheduled_briefings")